In [1]:
import cv2
import torch
import numpy as np
import faiss
from ultralytics import YOLO
from facenet_pytorch import InceptionResnetV1

# ==========================================
# 1. ИНИЦИАЛИЗАЦИЯ МОДЕЛЕЙ (PyTorch / YOLO)
# ==========================================
# Используем предобученную легкую YOLOv8 для детекции лиц
detector = YOLO('yolov8n-face.pt') # Скачается автоматически

# Сиамская сеть для эмбеддингов (выдает вектор 512)
embedder = InceptionResnetV1(pretrained='vggface2').eval()

# ==========================================
# 2. ПОДГОТОВКА БАЗЫ ДАННЫХ (Индекса FAISS)
# ==========================================
# Нам нужен словарь для связи ID в FAISS с именами и взводами
metadata = {}
db_vectors = []

def add_soldier_to_db(img_path, name, platoon_id):
    """Функция для регистрации солдата в базу данных по одному фото"""
    img = cv2.imread(img_path)
    # Детектируем лицо
    results = detector(img, verbose=False)
    if len(results[0].boxes) == 0:
        print(f"Предупреждение: Лицо для {name} не найдено!")
        return
    
    # Берем первое найденное лицо (координаты)
    box = results[0].boxes.xyxy[0].cpu().numpy().astype(int)
    face_crop = img[box[1]:box[3], box[0]:box[2]]
    
    # Предобработка для PyTorch модели ArcFace/FaceNet
    face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    face_resized = cv2.resize(face_rgb, (160, 160))
    face_tensor = torch.tensor(face_resized).permute(2, 0, 1).float() / 255.0
    face_tensor = (face_tensor - 0.5) / 0.5 # Нормализация
    
    # Получаем эмбеддинг
    with torch.no_grad():
        embedding = embedder(face_tensor.unsqueeze(0)).flatten().numpy()
    
    # Сохраняем вектор и метаданные
    db_vectors.append(embedding)
    idx = len(db_vectors) - 1
    metadata[idx] = {"name": name, "platoon": platoon_id}

# --- Симуляция заполнения базы данных ---
# В реальности здесь будут циклы по твоим папкам с фото
# add_soldier_to_db('ivanov.jpg', 'Рядовой Иванов', 'Взвод №1')
# add_soldier_to_db('petrov.jpg', 'Рядовой Петров', 'Взвод №1')
# add_soldier_to_db('sidorov.jpg', 'Сержант Сидоров', 'Взвод №2')

# Для теста создадим случайные векторы, как будто мы заполнили базу на 40 человек
np.random.seed(42)
fake_db = np.random.randn(40, 512).astype('float32')
faiss.normalize_L2(fake_db) # Нормализуем для косинусного расстояния

# Заполняем тестовые метаданные (20 человек во взводе 1, 20 человек во взводе 2)
for i in range(40):
    metadata[i] = {
        "name": f"Солдат {i}", 
        "platoon": "Взвод №1" if i < 20 else "Взвод №2"
    }

# Создаем индекс FAISS
index = faiss.IndexFlatIP(512) # Inner Product на нормализованных векторах = Косинусное сходство
index.add(fake_db)

# ==========================================
# 3. ОБРАБОТКА КАДРА С ПОСТРОЕНИЯ (Пайплайн)
# ==========================================
def process_platoon_building(frame_path, confidence_threshold=0.6):
    frame = cv2.imread(frame_path)
    if frame is None:
        # Для теста сгенерируем фейковый кадр, если файла нет
        print("Файл кадра не найден, запускаем симуляцию распознавания...")
        return simulate_detection()

    # Шаг 1: Детекция всех лиц на плацу
    results = detector(frame, verbose=False)
    boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
    
    current_embeddings = []
    
    # Шаг 2: Вырезание лиц и получение эмбеддингов
    for box in boxes:
        face_crop = frame[box[1]:box[3], box[0]:box[2]]
        if face_crop.size == 0: continue
            
        face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
        face_resized = cv2.resize(face_rgb, (160, 160))
        face_tensor = torch.tensor(face_resized).permute(2, 0, 1).float() / 255.0
        face_tensor = (face_tensor - 0.5) / 0.5
        
        with torch.no_grad():
            emb = embedder(face_tensor.unsqueeze(0)).flatten().numpy()
            current_embeddings.append(emb)
            
    if not current_embeddings:
        print("На кадре не обнаружено ни одного лица.")
        return

    current_embeddings = np.array(current_embeddings).astype('float32')
torch.no is parked
torch.no
faiss.normalize_L2(current_embeddings)
    
    # Шаг 3: Поиск в FAISS (Ищем ТОП-1 совпадение для каждого лица)
    similarities, indices = index.search(current_embeddings, k=1)
    
    # Шаг 4: Бизнес-логика (Мажоритарное голосование и проверка присутствия)
    recognized_people = []
    platoon_votes = {}
    
    for i, idx in enumerate(indices.flatten()):
        sim = similarities[i][0]
        if sim >= confidence_threshold: # Проверяем, что лицо действительно похоже, а не случайный шум
            person = metadata[idx]
            recognized_people.append(person)
            
            # Считаем голоса за взвод
            plat = person['platoon']
            platoon_votes[plat] = platoon_votes.get(plat, 0) + 1
        else:
            recognized_people.append({"name": "Неизвестный", "platoon": "Нет"})

    if not platoon_votes:
        print("Ни одно лицо не опознано по базе данных.")
        return

    # Определяем, какой взвод на построении
    detected_platoon = max(platoon_votes, key=platoon_votes.get)
    print(f"\n=== ВЕРДИКТ СИСТЕМЫ ===")
    print(f"На плацу обнаружен: **{detected_platoon}** (Голосов: {platoon_votes[detected_platoon]} из {len(current_embeddings)})")
    
    # Проверяем, кого не хватает из этого взвода
    all_platoon_soldiers = [meta["name"] for meta in metadata.values() if meta["platoon"] == detected_platoon]
    present_soldiers = [p["name"] for p in recognized_people if p["platoon"] == detected_platoon]
    absent_soldiers = list(set(all_platoon_soldiers) - set(present_soldiers))
    
    print(f"\nПрисутствуют ({len(present_soldiers)}): {', '.join(present_soldiers)}")
    print(f"Отсутствуют ({len(absent_soldiers)}): {', '.join(absent_soldiers) if absent_soldiers else 'Все на месте!'}")

def simulate_detection():
    """Вспомогательная функция для демонстрации логики без реальной камеры"""
    # Имитируем, что мы извлекли 18 векторов (15 из Взвода 1, 1 из Взвода 2, 2 неизвестных)
    print("Имитация: Камера зафиксировала 18 человек.")
    simulated_indices = list(range(15)) + [25] + [-1, -1] 
    
    platoon_votes = {}
    present = []
    
    for idx in simulated_indices:
        if idx != -1:
            person = metadata[idx]
            plat = person['platoon']
            platoon_votes[plat] = platoon_votes.get(plat, 0) + 1
            if idx < 15: present.append(person['name']) # Только для 1 взвода
            
    detected_platoon = max(platoon_votes, key=platoon_votes.get)
    all_platoon_soldiers = [meta["name"] for meta in metadata.values() if meta["platoon"] == detected_platoon]
    absent = list(set(all_platoon_soldiers) - set(present))
    
    print(f"\n=== ВЕРДИКТ СИСТЕМЫ (ТЕСТ) ===")
    print(f"На плацу обнаружен: **{detected_platoon}**")
    print(f"Присутствуют ({len(present)}): {', '.join(present)}")
    print(f"Отсутствуют ({len(absent)}): {', '.join(absent)}")

# Запуск теста
process_platoon_building('test_platoon.jpg')

IndentationError: unexpected indent (745886447.py, line 114)